# 언어 모델링 단계별 문제은행

이 노트북은 `07_language_modeling`의 세 수업 노트북을 바탕으로 언어 모델의 공통 흐름, N-gram, feed-forward NNLM, GPT-2 추론을 총 30문제로 복습합니다.

- 각 문제 아래의 빈 코드 셀에 직접 풀이하세요.
- 설명 문제는 코드 셀에 주석으로 답해도 됩니다.
- 앞 문제에서 만든 변수를 뒤 문제에서 다시 사용하므로 위에서부터 실행하는 것을 권장합니다.
- **1단계**는 개념과 작은 텐서, **2단계**는 Bigram, **3단계**는 NNLM, **4단계**는 GPT-2, **5단계**는 디버깅과 종합 문제입니다.
- 4단계는 `transformers`와 최초 실행 시 모델 다운로드가 필요합니다. 나머지 단계는 GPT-2 없이도 풀 수 있습니다.
- 정답을 보기 전에는 문제의 `검증` 항목을 `assert` 또는 출력으로 먼저 확인해 보세요.
- 풀이 후 같은 폴더의 `정답및해설.md`에서 핵심 코드와 예상값을 확인할 수 있습니다.

난이도 표기: `★` 개념 확인 · `★★` 기본 구현 · `★★★` 응용 · `★★★★` 종합

## 원본 노트북과 문제 연결

- `01_language_modeling.ipynb`: 문제 1~6, 22~27, 30
- `02_ngram.ipynb`: 문제 7~13, 28, 30
- `03_nnlm.ipynb`: 문제 14~21, 29, 30

| 단계 | 문제 | 핵심 주제 |
| --- | ---: | --- |
| 1. 공통 원리 | 1~6 | 연쇄법칙, 학습·추론, 로짓·확률, shape, decoding |
| 2. N-gram | 7~13 | 경계 토큰, 빈도, MLE, Add-k, perplexity, 생성 |
| 3. NNLM | 14~21 | 학습 쌍, Embedding, forward, CE loss, 역전파, 생성 |
| 4. GPT-2 | 22~27 | 사전학습 모델, tokenizer, top-k, greedy, sampling |
| 5. 종합 | 28~30 | 오류 수정, 모델 비교, 전체 파이프라인 설계 |

In [ ]:
from collections import Counter
import math
import random

import torch
from torch import nn

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

ngram_sentences = [
    "나는 자연어 처리를 공부한다",
    "나는 자연어 처리를 배운다",
    "학생은 자연어 처리를 공부한다",
    "학생은 언어 모델을 공부한다",
    "나는 언어 모델을 공부한다",
    "언어 모델은 다음 단어를 예측한다",
    "자연어 처리는 문장을 분석한다",
    "머신러닝 모델은 데이터를 학습한다",
    "학생은 머신러닝 모델을 배운다",
    "나는 다음 단어를 예측한다",
]

nnlm_sentences = [
    ["나는", "자연어", "처리를", "공부한다"],
    ["나는", "자연어", "처리를", "배운다"],
    ["나는", "자연어", "처리를", "좋아한다"],
    ["나는", "언어", "모델을", "공부한다"],
    ["나는", "언어", "모델을", "배운다"],
    ["나는", "머신러닝", "모델을", "공부한다"],
    ["학생은", "자연어", "처리를", "공부한다"],
    ["학생은", "언어", "모델을", "배운다"],
    ["언어", "모델은", "다음", "단어를", "예측한다"],
    ["자연어", "처리는", "문장을", "분석한다"],
    ["머신러닝", "모델은", "데이터를", "학습한다"],
]

START_TOKEN = "<s>"
END_TOKEN = "</s>"
OOV_TOKEN = "<OOV>"

print("Bigram 문장 수:", len(ngram_sentences))
print("NNLM 문장 수:", len(nnlm_sentences))

# 1단계. 언어 모델의 공통 원리

## 문제 1. 문장 확률을 조건부확률로 분해하기 ★

토큰 시퀀스 `<BOS> 나는 자연어를 공부한다 <EOS>`를 생각하세요.

- 연쇄법칙을 사용해 문장 확률을 위치별 조건부확률의 곱으로 작성하세요.
- `<EOS>`의 확률까지 포함해야 하는 이유를 설명하세요.
- 네 번의 다음 토큰 선택 확률이 각각 `[0.6, 0.5, 0.4, 0.8]`이라고 가정하고 전체 확률을 계산하세요.
- 문장이 길어질수록 확률의 곱이 작아지는 이유와, 길이가 다른 문장을 문장 확률만으로 비교하면 불공정할 수 있는 이유를 작성하세요.

검증: 계산 결과는 `0 < sentence_probability < 1`이어야 합니다.

## 문제 2. 학습과 추론 구분하기 ★

다음 항목을 학습과 추론으로 나누어 표 또는 주석으로 정리하세요.

1. 실제 다음 토큰 ID가 주어진다.
2. `loss.backward()`와 `optimizer.step()`을 실행한다.
3. 모델이 고른 토큰을 다음 문맥에 다시 넣는다.
4. `model.eval()`과 `torch.no_grad()`를 사용한다.
5. 가중치가 갱신된다.
6. 앞에서 잘못 고른 토큰이 뒤의 예측에도 영향을 줄 수 있다.

추가 질문:

- `from_pretrained()`로 가중치를 불러와 문장을 생성한 것을 현재 노트북에서 새로 학습했다고 말할 수 없는 이유는 무엇인가요?
- Teacher forcing을 사용하는 학습과 자동회귀 추론의 입력 차이를 한 문장으로 설명하세요.

## 문제 3. 로짓을 다음 토큰 확률로 바꾸기 ★★

아래 로짓은 배치 2개의 다음 토큰 후보 4개에 대한 정규화 전 점수입니다.

```python
logits = torch.tensor([
    [2.0, 1.0, 0.0, -1.0],
    [0.2, 0.2, 0.2, 0.2],
])
vocabulary = ["공부한다", "배운다", "좋아한다", "</s>"]
```

- `torch.softmax(..., dim=-1)`로 `probabilities`를 만드세요.
- 각 행의 확률 합이 1인지 확인하세요.
- `torch.topk()`로 각 배치의 top-2 확률과 ID를 구하고 문자열 토큰으로 복원하세요.
- 두 번째 행의 분포가 균등한 이유를 설명하세요.
- 로짓에 softmax를 적용하지 않은 값 자체를 확률이라고 부르면 안 되는 이유를 작성하세요.

## 문제 4. 언어 모델의 Tensor shape 추적하기 ★★

배치 크기 `B=3`, 입력 길이 `T=5`, 어휘 크기 `V=20`인 causal language model이 있다고 가정합니다.

- 입력 토큰 ID, 전체 위치의 로짓, 마지막 위치 로짓, 다음 토큰 확률, top-4 값과 ID의 shape를 각각 작성하세요.
- `fake_logits = torch.randn(B, T, V)`를 만들고 마지막 위치만 선택해 실제 shape를 출력하세요.
- `fake_logits[:, -1, :]`에서 `-1`이 어휘의 마지막 후보를 뜻하는지, 입력 위치의 마지막을 뜻하는지 설명하세요.
- 문장 바로 다음 토큰 하나를 예측할 때 전체 `(B, T, V)` 중 마지막 위치를 사용하는 이유를 작성하세요.

검증: 마지막 위치 로짓과 확률의 shape는 모두 `(3, 20)`이어야 합니다.

## 문제 5. Greedy와 sampling 비교하기 ★★

다음 토큰 분포가 아래와 같다고 가정합니다.

```python
tokens = ["자연어", "언어", "모델", "데이터"]
probabilities = torch.tensor([0.45, 0.30, 0.15, 0.10])
```

- greedy 방식으로 선택한 토큰을 구하세요.
- `torch.multinomial()`로 sampling을 10회 수행하고 결과를 출력하세요. 재현을 위해 seed를 지정하세요.
- greedy 결과가 같은 입력에서 보통 일정한 이유를 설명하세요.
- sampling에서 확률이 낮은 후보도 선택될 수 있는 이유를 설명하세요.
- 생성 품질의 사실성·안전성이 두 선택 방식만으로 보장되지 않는 이유를 한 문장으로 작성하세요.

## 문제 6. Temperature가 분포에 주는 영향 ★★★

아래 로짓을 `temperature`가 `0.5`, `1.0`, `2.0`일 때 각각 `softmax(logits / temperature)`로 변환하세요.

```python
temperature_logits = torch.tensor([2.5, 1.5, 0.5, -0.5])
```

- 온도별 확률 벡터와 최댓값을 출력하세요.
- 모든 분포의 합이 1인지 확인하세요.
- 온도가 낮을 때와 높을 때 어느 분포가 더 뾰족한지 설명하세요.
- temperature를 로짓에 곱하는 것이 아니라 나누는 일반적인 구현을 작성하세요.
- `temperature=0`을 그대로 나눗셈에 사용하면 안 되는 이유를 설명하세요.

# 2단계. N-gram 언어 모델

## 문제 7. 어휘·경계 토큰·OOV 처리하기 ★★

설정 셀의 `ngram_sentences`로 Bigram 학습 입력을 준비하세요.

- 공백 기준으로 토큰화한 `ngram_tokenized`를 만드세요.
- 중복 없는 학습 어휘에 `<s>`, `</s>`, `<OOV>`를 추가해 `ngram_vocabulary`를 만드세요.
- 학습 어휘 밖의 토큰을 `<OOV>`로 바꾸고 양끝에 경계 토큰을 붙이는 `add_boundaries(tokens)`를 작성하세요.
- 모든 학습 문장을 변환한 `train_sequences`를 만드세요.
- `연구자는 자연어를 평가한다`를 변환해 OOV 위치를 확인하세요.

검증:

- 첫 학습 시퀀스의 첫 토큰은 `<s>`, 마지막 토큰은 `</s>`여야 합니다.
- `<OOV>`는 패딩이 아니며 하나의 예측 가능한 어휘 항목임을 설명하세요.

## 문제 8. Bigram과 문맥 빈도 세기 ★★

문제 7의 `train_sequences`에서 빈도표를 만드세요.

- `Counter`를 사용해 `(previous_token, next_token)` 쌍의 빈도 `bigram_counts`를 구하세요.
- 이전 토큰별 전체 출발 횟수 `context_counts`를 구하세요.
- `나는` 뒤에 관찰된 모든 다음 토큰과 빈도를 빈도 내림차순·토큰 오름차순으로 출력하세요.
- `context_counts["나는"]`와 `나는`에서 출발한 Bigram 빈도의 합이 같은지 `assert`로 확인하세요.
- N-gram에서는 어떤 값이 학습된 모델로 남는지 설명하세요.

## 문제 9. 최대우도 Bigram 확률 구현하기 ★★

`P(next | previous) = count(previous, next) / count(previous, *)`를 구현하세요.

- 함수 이름은 `bigram_probability(previous_token, next_token)`으로 하세요.
- 분모가 0이면 `0.0`을 반환하세요.
- `나는` 뒤의 관찰 후보별 빈도와 확률을 출력하세요.
- `나는` 뒤 전체 관찰 후보의 확률 합이 1인지 확인하세요.
- `P(문장을 | 나는)`를 계산하고 결과의 근거가 되는 분자와 분모를 함께 출력하세요.

추가 질문: 이 확률을 Bayesian posterior가 아니라 빈도 기반 최대우도 추정이라고 부르는 이유를 간단히 설명하세요.

## 문제 10. 미관찰 Bigram과 문장 확률 0 찾기 ★★★

다음 세 문장을 경계 토큰까지 포함한 Bigram 목록으로 바꾸고, 각 전이의 확률을 출력하세요.

```python
evaluation_sentences = {
    "학습 문장": "나는 자연어 처리를 공부한다",
    "held-out 문장": "나는 자연어 처리를 예측한다",
    "OOV 문장": "연구자는 자연어 모델을 평가한다",
}
```

- 문장 확률을 모든 Bigram 확률의 곱으로 계산하는 함수를 작성하세요.
- 확률이 0인 최초 전이를 문장별로 찾아 출력하세요.
- 단어가 모두 학습 어휘에 있어도 held-out 문장의 확률이 0이 될 수 있는 이유를 설명하세요.
- OOV 치환이 되었더라도 `<OOV>` 관련 전이를 학습에서 보지 못했다면 어떤 문제가 남는지 설명하세요.

## 문제 11. Add-k smoothing 구현과 정규화 검증 ★★★

모든 다음 토큰 후보의 빈도에 `k=0.1`을 더하는 Add-k smoothing을 구현하세요.

- 예측 후보 어휘 `prediction_vocabulary`에서는 시작 토큰 `<s>`를 제외하세요.
- 함수 이름은 `smoothed_bigram_probability(previous_token, next_token, k=0.1)`으로 하세요.
- 분자는 `count + k`, 분모는 `context_count + k × |V_next|`를 사용하세요.
- 미관찰 쌍 `("나는", "문장을")`의 smoothing 전후 확률을 비교하세요.
- `나는` 뒤 모든 예측 후보의 평활 확률 합이 1인지 확인하세요.
- 관찰된 Bigram의 확률도 smoothing 후 변하는 이유를 설명하세요.

검증: `abs(probability_sum - 1.0) < 1e-9`를 만족해야 합니다.

## 문제 12. 로그확률로 perplexity 계산하기 ★★★

문제 10의 세 평가 문장에 대해 Bigram perplexity를 계산하세요.

- `PPL = exp(-(1/T) × Σ log P(next | previous))`를 구현하세요.
- 함수 이름은 `sentence_perplexity(sentence, use_smoothing)`으로 하세요.
- 확률이 하나라도 0이면 smoothing 전 PPL은 `math.inf`를 반환하세요.
- `<s>` 다음 첫 토큰부터 `</s>`까지의 전이 수를 `T`로 사용하세요.
- smoothing 전후 결과를 나란히 출력하세요.
- 문장 확률을 직접 계속 곱하는 대신 로그확률을 더하는 수치적 이유를 설명하세요.
- PPL은 낮을수록 무엇을 뜻하며, 서로 다른 tokenizer·어휘 조건의 PPL을 단순 비교하면 안 되는 이유를 작성하세요.

## 문제 13. Bigram 빈도로 문장 생성하기 ★★★

관찰한 Bigram 빈도를 sampling 가중치로 사용해 문장을 생성하세요.

- `<s>`에서 시작해 현재 토큰 뒤의 관찰 후보와 빈도를 모으세요.
- `random.Random(seed).choices(..., weights=..., k=1)`로 다음 토큰을 뽑으세요.
- `</s>` 또는 `max_tokens`에서 종료하는 `generate_bigram_sentence(max_tokens=12, seed=7)`를 작성하세요.
- seed `3`, `7`, `11`의 결과를 비교하세요.
- 생성 결과에 나타난 모든 인접 쌍이 `bigram_counts`에 실제로 존재하는지 검사하는 함수를 작성하세요.
- 이 생성 결과가 문법과 의미를 별도로 이해한 증거가 아닌 이유를 설명하세요.

# 3단계. Feed-forward NNLM

## 문제 14. 고정 길이 문맥-정답 학습 쌍 만들기 ★★

설정 셀의 `nnlm_sentences`와 문맥 길이 `CONTEXT_SIZE=2`를 사용하세요.

- 코퍼스 토큰을 정렬하고 `[<s>, </s>, <OOV>, ...]` 순서의 `index_to_word`를 만드세요.
- 역방향 사전 `word_to_index`를 만드세요.
- 문장 앞에 `<s>`를 두 번, 뒤에 `</s>`를 한 번 붙이세요.
- 각 정답 직전 두 토큰을 입력으로 하는 `training_examples`를 만드세요.
- 입력을 `context_ids` shape `(B, 2)`, 정답을 `target_ids` shape `(B,)`인 `torch.long` Tensor로 바꾸세요.
- 첫 다섯 쌍을 문자열과 ID로 모두 출력하세요.

검증: 수업 코퍼스를 그대로 사용했다면 어휘 크기는 22, 학습 쌍 수는 56이어야 합니다.

## 문제 15. Feed-forward NNLM 정의와 shape 추적 ★★

다음 구조의 `FeedForwardNNLM(nn.Module)`을 작성하세요.

```text
token IDs (B, 2)
→ Embedding (B, 2, 8)
→ Flatten (B, 16)
→ Linear + tanh (B, 32)
→ Linear logits (B, V)
```

- `nn.Embedding`, `nn.Linear`, `torch.tanh`를 사용하세요.
- forward 안에서 softmax를 적용하지 마세요.
- 전체 `context_ids`를 통과시켜 최종 로짓 shape를 확인하세요.
- 두 임베딩을 평균내지 않고 이어 붙이면 토큰 순서가 어떤 방식으로 보존되는지 설명하세요.

검증: 출력 로짓 shape는 `(56, 22)`여야 합니다.

## 문제 16. 계층별 파라미터 수 계산하기 ★★★

문제 15 모델의 학습 파라미터를 계층별로 계산하세요.

- Embedding 가중치 shape와 원소 수를 출력하세요.
- hidden 계층의 weight·bias shape와 원소 수를 출력하세요.
- output 계층의 weight·bias shape와 원소 수를 출력하세요.
- 직접 계산한 합과 `sum(p.numel() for p in model.parameters())`가 같은지 확인하세요.
- Embedding의 한 행과 토큰 ID의 관계를 설명하세요.
- `embedding.weight`의 값이 빈도나 확률이 아니라는 점을 설명하세요.

## 문제 17. 학습 전 top-k 분포 저장하기 ★★

문맥 `[<s>, 나는]`에서 학습 전 다음 토큰 분포를 확인하세요.

- 문자열 문맥을 ID Tensor shape `(1, 2)`로 바꾸세요. 미등록 문자열은 `<OOV>` ID로 처리하세요.
- 평가 모드와 `torch.no_grad()`에서 로짓을 계산하세요.
- softmax 후 top-5 토큰과 확률을 문자열로 복원하는 `top_k_next_tokens()`를 작성하세요.
- 목표 토큰 `자연어`의 확률을 반환하는 `next_token_probability()`를 작성하세요.
- 학습 전 top-5와 목표 확률을 `initial_top_k`, `initial_target_probability`에 저장하세요.
- 이 시점의 분포가 코퍼스 빈도와 맞지 않아도 정상인 이유를 설명하세요.

## 문제 18. CrossEntropyLoss로 NNLM 학습하기 ★★★

문제 15의 모델을 전체 배치로 300 epoch 학습하세요.

- `nn.CrossEntropyLoss()`와 `torch.optim.Adam(..., lr=0.03)`을 사용하세요.
- 각 epoch에서 `zero_grad → forward → loss → backward → step` 순서를 지키세요.
- loss를 `loss_history`에 저장하고 1, 50, 100, ..., 300 epoch의 loss와 `exp(loss)`를 출력하세요.
- 처음과 마지막 loss가 감소했는지 확인하세요.
- 모델 출력에 softmax를 먼저 적용하지 않는 이유를 설명하세요.
- 현재 `exp(loss)`가 학습 코퍼스의 지표이지 새로운 문장에 대한 일반화 성능은 아닌 이유를 작성하세요.

## 문제 19. 임베딩이 실제로 학습되었는지 검증하기 ★★★

토큰 `자연어`의 임베딩을 학습 전후로 비교하세요. 문제 18을 다시 실행해야 한다면 학습 전에 초기 벡터를 반드시 복사해 두세요.

- `detach().clone()`으로 초기 임베딩을 저장하세요.
- 학습 후 같은 ID의 임베딩을 다시 가져오세요.
- 앞 4개 값을 나란히 출력하고 차이의 L2 norm을 계산하세요.
- 변화량이 0보다 큰지 확인하세요.
- 단순히 `initial_embedding = model.embedding.weight[id]`로 참조만 저장하면 비교가 잘못될 수 있는 이유를 설명하세요.
- 역전파가 출력층에서 임베딩까지 도달하는 경로를 순서대로 작성하세요.

## 문제 20. 학습 전후 다음 토큰 확률 비교하기 ★★★

문제 17의 동일한 문맥 `[<s>, 나는]`을 다시 평가하세요.

- 학습 후 top-5와 `자연어` 확률을 구하세요.
- 학습 전후 top-5를 나란히 출력하세요.
- `자연어` 확률의 변화량을 계산하세요.
- 코퍼스에서 `[<s>, 나는]` 뒤에 실제로 등장한 다음 토큰별 빈도와 경험적 확률을 구하세요.
- NNLM의 예측 분포와 경험적 빈도 분포가 완전히 같아야 하는지 설명하세요.
- 같은 문맥에 여러 정답이 있으면 한 후보가 확률 1을 가질 필요가 없는 이유를 작성하세요.

## 문제 21. NNLM 자동회귀 생성과 고정 문맥의 한계 ★★★

`<s>`와 첫 토큰에서 시작하는 greedy 생성 함수 `generate_with_nnlm(model, first_token, max_new_tokens=8)`을 작성하세요.

- 현재 두 토큰 문맥에서 top-1 다음 토큰을 선택하세요.
- 새 토큰을 붙인 뒤 `[이전 문맥의 마지막 토큰, 새 토큰]`으로 문맥을 이동하세요.
- `</s>` 또는 최대 생성 횟수에서 종료하세요.
- 첫 토큰 `나는`, `학생은`, 미등록 토큰 `연구자는`의 결과를 비교하세요.
- 미등록 첫 토큰은 어떤 ID로 바뀌는지 출력하세요.
- 최근 두 토큰보다 먼 정보가 직접 입력에서 사라지는 고정 문맥창의 한계를 설명하세요.
- 같은 OOV ID로 바뀐 서로 다른 문자열을 모델이 구분할 수 있는지 답하세요.

# 4단계. 사전학습 GPT-2 추론

## 문제 22. GPT-2 tokenizer와 모델 불러오기 ★★

`openai-community/gpt2`의 사전학습 tokenizer와 causal language model을 불러오세요.

- `AutoTokenizer`와 `AutoModelForCausalLM`을 사용하세요.
- 모델을 평가 모드로 바꾸세요.
- checkpoint 이름, 어휘 크기, EOS 토큰과 ID를 출력하세요.
- GPT-2에는 기본 PAD 토큰이 없으므로 생성용 `pad_token_id`에 EOS ID를 지정하세요.
- `Generative`, `Pre-trained`, `Transformer`가 각각 뜻하는 바를 한 문장씩 작성하세요.
- 현재 셀에서 optimizer와 역전파가 없다는 점을 근거로 이 작업이 학습인지 추론인지 답하세요.

주의: 최초 실행 시 모델 파일 다운로드가 필요합니다.

## 문제 23. GPT-2 입력 토큰과 전체 로짓 shape 읽기 ★★

프롬프트 `Natural language processing helps computers`를 Tensor 입력으로 변환해 모델에 전달하세요.

- tokenizer 결과의 key와 각 Tensor shape를 출력하세요.
- 입력 ID를 토큰 조각 문자열로 복원해 위치별 ID와 함께 출력하세요.
- `torch.no_grad()`에서 모델을 실행하세요.
- 전체 로짓의 shape `(B, T, V)`를 출력하고 실제 `B`, `T`, `V`를 해석하세요.
- 각 입력 위치의 로짓이 어느 토큰을 예측하기 위한 값인지 설명하세요.
- `model.eval()`과 `torch.no_grad()`의 역할 차이를 설명하세요.

## 문제 24. GPT-2 다음 토큰 top-k 복원하기 ★★★

문제 23의 모델 출력에서 프롬프트 바로 다음 토큰 후보를 구하세요.

- 마지막 입력 위치의 로짓만 `last_token_logits`로 선택하세요.
- softmax로 `next_token_probabilities`를 만드세요.
- 확률 합이 1인지 확인하세요.
- top-5 확률과 ID를 구해 토큰 문자열로 복원하세요.
- 각 후보의 순위, `repr(token_text)`, ID, 확률을 출력하세요.
- GPT-2 토큰 조각 앞의 공백도 문자열 정보의 일부일 수 있으므로 `repr()`이 유용한 이유를 설명하세요.
- top-1만 출력하는 것보다 top-k를 함께 보는 장점을 작성하세요.

## 문제 25. Greedy 생성 결과 분리해서 읽기 ★★★

문제 23의 프롬프트에서 greedy 방식으로 최대 24개 새 토큰을 생성하세요.

- `model.generate()`에 `do_sample=False`, `max_new_tokens=24`, `pad_token_id`, `repetition_penalty=1.1`을 지정하세요.
- 결과 Tensor의 shape를 출력하세요.
- 전체 결과와 새로 생성된 ID 부분만 각각 분리해 출력하세요.
- 전체 텍스트와 새 텍스트만 따로 decode하세요.
- `max_new_tokens`와 전체 출력 길이의 관계를 설명하세요.
- `skip_special_tokens=True`가 모델 계산에서 특수 토큰을 삭제하는 옵션인지, 화면 문자열 복원에서 제외하는 옵션인지 구분하세요.

## 문제 26. Sampling 설정 비교하기 ★★★

같은 프롬프트와 모델에서 다음 세 설정을 비교하세요.

```python
sampling_configs = [
    {"temperature": 0.7, "top_k": 40, "top_p": 0.9},
    {"temperature": 1.0, "top_k": 40, "top_p": 0.9},
    {"temperature": 1.2, "top_k": 0,  "top_p": 0.95},
]
```

- 각 설정에 `do_sample=True`, `max_new_tokens=24`를 추가해 생성하세요.
- 각 실행 전 같은 seed를 다시 지정한 비교와, 서로 다른 seed를 지정한 비교를 모두 수행하세요.
- `temperature`, `top_k`, `top_p`가 각각 후보 분포를 어떻게 바꾸는지 설명하세요.
- `top_k`와 `top_p`를 동시에 적용하면 둘 중 하나만 무조건 사용되는지, 필터가 함께 적용되는지 확인해 설명하세요.
- sampling 결과가 달라졌다고 모델 가중치가 바뀐 것은 아닌 이유를 작성하세요.

## 문제 27. Causal language modeling의 위치별 정답 만들기 ★★★

짧은 프롬프트 `Language models predict tokens`를 토큰화하세요.

- 입력 ID의 `[:-1]`을 예측 입력 위치, `[1:]`을 다음 토큰 정답으로 나란히 출력하세요.
- 각 위치에서 왼쪽 문맥과 바로 다음 정답 토큰을 문자열로 표시하세요.
- 미래 토큰을 보지 못하게 하는 causal mask가 필요한 이유를 설명하세요.
- BERT의 masked language modeling과 GPT의 causal language modeling을 `보는 문맥`, `학습 정답`, `일반적 생성 방식`으로 비교하세요.
- 실제 `AutoModelForCausalLM`에 `labels=input_ids`를 전달하면 내부에서 위치를 맞춰 loss를 계산한다는 점과, 추론 시에는 labels가 필요 없다는 점을 설명하세요.

# 5단계. 종합·디버깅

## 문제 28. Bigram 코드의 네 가지 오류 수정하기 ★★★★

다음 코드에는 조건부확률과 Add-k smoothing에 관한 오류가 네 가지 이상 있습니다. 올바르게 수정하고 각 오류의 이유를 설명하세요.

```python
prediction_vocabulary = sorted(ngram_vocabulary)  # 그대로 사용

def broken_bigram_probability(previous_token, next_token):
    denominator = bigram_counts[(previous_token, next_token)]
    if denominator == 0:
        return 0.0
    return context_counts[previous_token] / denominator

def broken_smoothed_probability(previous_token, next_token, k=0.1):
    numerator = bigram_counts[(previous_token, next_token)] + k
    denominator = context_counts[previous_token] + len(prediction_vocabulary)
    return denominator / numerator
```

검증:

- 수정한 일반 Bigram 확률은 `나는` 뒤 관찰 후보에 대해 합이 1이어야 합니다.
- 수정한 평활 확률은 전체 예측 후보에 대해 합이 1이어야 합니다.
- 시작 토큰은 다음 토큰 후보 어휘에서 제외해야 합니다.

## 문제 29. NNLM 학습 코드의 shape·손실 오류 수정하기 ★★★★

다음 코드가 올바르게 학습되지 않는 이유를 찾고 수정하세요.

```python
class BrokenNNLM(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, 8)
        self.hidden = nn.Linear(8, 32)
        self.output = nn.Linear(32, vocabulary_size)

    def forward(self, token_ids):
        embedded = self.embedding(token_ids)       # (B, 2, 8)
        hidden = torch.tanh(self.hidden(embedded))
        probabilities = torch.softmax(self.output(hidden), dim=1)
        return probabilities

broken_model = BrokenNNLM(len(index_to_word))
predictions = broken_model(context_ids)
loss = nn.CrossEntropyLoss()(predictions, target_ids)
loss.backward()
```

- 두 문맥 임베딩을 `(B, 16)`으로 만드는 처리를 추가하세요.
- hidden 계층 입력 크기를 수정하세요.
- 출력은 `(B, V)` 로짓이어야 합니다.
- CrossEntropyLoss 앞의 softmax를 제거하세요.
- softmax의 어휘 축을 굳이 쓴다면 어느 축인지 답하세요.
- 수정 모델로 한 번의 optimizer step을 실행하고 모든 주요 shape를 출력하세요.

## 문제 30. 최종 종합: 세 언어 모델의 공통 흐름과 선택 기준 ★★★★

N-gram, feed-forward NNLM, GPT-2를 아래 기준으로 비교표에 정리하세요.

| 비교 기준 | N-gram | Feed-forward NNLM | GPT-2 |
| --- | --- | --- | --- |
| 문맥을 표현하는 방법 | | | |
| 학습 결과로 저장되는 핵심 값 | | | |
| 다음 토큰 점수·확률 계산 | | | |
| 미관찰 표현 처리 | | | |
| 처리 가능한 문맥 범위 | | | |
| 학습·추론 비용 | | | |
| 대표 한계 | | | |

그리고 다음 종합 과제를 수행하세요.

1. 세 모델 모두에서 `문맥 → 어휘별 점수 또는 빈도 → 확률 → 토큰 선택 → 문맥 갱신` 흐름이 어디에 해당하는지 작성하세요.
2. 작은 폐쇄형 명령어 데이터에서 확률 근거를 사람이 직접 확인해야 하는 경우, 세 모델 중 첫 기준선을 고르고 이유를 쓰세요.
3. 대규모 일반 문장 생성이 필요한 경우의 후보를 고르고, 사실성·편향·비용 측면의 위험을 쓰세요.
4. 모델 비교 실험을 설계한다면 같은 train/validation 분리, tokenizer·어휘 조건, 평가 지표를 어떻게 통제할지 작성하세요.
5. PPL 외에 생성 결과를 평가할 방법을 최소 세 가지 제안하세요.
6. 마지막으로 수업 전체를 다음 한 줄의 변수명과 대표 shape를 포함해 설명하세요.

```text
문자열 → token IDs → context representation → logits → probabilities → selected token ID → 문자열
```

## 마무리 점검표

아래 항목을 설명 없이 코드만 외우지 않고 말로 설명할 수 있는지 확인하세요.

- [ ] 문장 확률을 다음 토큰 조건부확률의 곱으로 표현할 수 있다.
- [ ] 로짓, softmax 확률, top-k ID의 차이와 shape를 설명할 수 있다.
- [ ] N-gram의 빈도·Add-k smoothing·perplexity가 연결되는 과정을 설명할 수 있다.
- [ ] NNLM의 ID → Embedding → Flatten → hidden → logits 경로를 추적할 수 있다.
- [ ] CrossEntropyLoss에 softmax 결과가 아니라 로짓을 넣는 이유를 설명할 수 있다.
- [ ] 학습과 추론, greedy와 sampling을 구분할 수 있다.
- [ ] GPT-2의 `(B, T, V)` 로짓에서 마지막 위치의 다음 토큰 후보를 복원할 수 있다.
- [ ] 세 모델의 공통점과 문맥 표현 방식의 차이를 비교할 수 있다.